# Más allá de la tabla · Notebook 2: estadísticas por equipo y temporada

**Objetivo:** transformar los partidos en estadísticas de cada equipo por temporada, separando su rendimiento como **local** y como **visitante**, y calcular la clasificación a partir de los resultados.

**Entrada:** `partidos_limpios.csv`, generado por el notebook 1.

**Tres niveles de datos.** Es importante no mezclarlos:

| Nivel | Qué representa una fila | Filas | Archivo |
|---|---|---|---|
| Partido | Un partido, con datos del local y del visitante | 1.900 | `partidos_limpios.csv` |
| Equipo-partido | Lo que hizo **un equipo** en **un partido** | 3.800 | `equipo_partido.csv` |
| Equipo-temporada | La suma de los 38 partidos de un equipo (o sus 19 de local, o sus 19 de visitante) | 300 | `estadisticas_equipo_temporada.csv` |

La pregunta guía se analiza en el nivel **equipo-temporada**, que es la unidad de análisis acordada.

## Paso 0. Cargar los partidos limpios

Suban `partidos_limpios.csv` al panel de archivos de Colab. Es el archivo que descargaron al terminar el notebook 1.

In [ ]:
from pathlib import Path

import pandas as pd

ARCHIVO_PARTIDOS = Path("partidos_limpios.csv")

if not ARCHIVO_PARTIDOS.exists():
    raise FileNotFoundError("Suban partidos_limpios.csv (lo genera el notebook 1).")

partidos = pd.read_csv(ARCHIVO_PARTIDOS, parse_dates=["Date"])

# Identificador de cada partido, para poder unir los niveles más adelante.
partidos.insert(0, "PartidoID", range(len(partidos)))

assert partidos.shape == (1900, 23), "No es el archivo que generó el notebook 1"
print(f"Partidos cargados: {len(partidos)}")
partidos.head()

Agregamos `PartidoID`, un número único por partido. Sirve para comprobar después que los datos del local y del visitante de un mismo partido son coherentes entre sí.

## Paso 1. De partido a equipo-partido

Cada partido tiene dos protagonistas, así que lo convertimos en **dos filas**: una para el local y otra para el visitante. Cada fila describe lo que hizo ese equipo: sus goles a favor y en contra, sus tiros, córners, faltas y tarjetas, y los puntos que ganó.

Usamos dos **diccionarios** que dicen de qué columna sale cada dato según la condición. Por ejemplo, los tiros del local están en `HS` y los del visitante en `AS`.

In [ ]:
# Diccionarios: nombre nuevo -> columna original, según la condición del equipo.
COLUMNAS_LOCAL = {
    "Equipo": "HomeTeam", "Rival": "AwayTeam",
    "GolesFavor": "FTHG", "GolesContra": "FTAG",
    "Tiros": "HS", "TirosArco": "HST", "Corners": "HC",
    "Faltas": "HF", "Amarillas": "HY", "Rojas": "HR",
}

COLUMNAS_VISITANTE = {
    "Equipo": "AwayTeam", "Rival": "HomeTeam",
    "GolesFavor": "FTAG", "GolesContra": "FTHG",
    "Tiros": "AS", "TirosArco": "AST", "Corners": "AC",
    "Faltas": "AF", "Amarillas": "AY", "Rojas": "AR",
}


def puntos_y_resultado(goles_favor, goles_contra):
    """Calcula los puntos y el resultado de un equipo en un partido.

    Args:
        goles_favor: goles que marcó el equipo.
        goles_contra: goles que recibió el equipo.

    Returns:
        Tupla (puntos, resultado): (3, "V") si ganó, (1, "E") si empató
        y (0, "D") si perdió.
    """
    if goles_favor > goles_contra:
        return 3, "V"
    elif goles_favor == goles_contra:
        return 1, "E"
    else:
        return 0, "D"


def filas_de_un_partido(fila):
    """Convierte un partido en dos registros: uno para el local y otro para el visitante.

    Args:
        fila: una fila de la tabla de partidos.

    Returns:
        Lista con dos diccionarios, uno por equipo.
    """
    registros = []
    for condicion, columnas in [("Local", COLUMNAS_LOCAL), ("Visitante", COLUMNAS_VISITANTE)]:
        registro = {
            "PartidoID": fila["PartidoID"],
            "Temporada": fila["Temporada"],
            "Fecha": fila["Date"],
            "Condicion": condicion,
        }
        for nombre_nuevo, columna_original in columnas.items():
            registro[nombre_nuevo] = fila[columna_original]

        puntos, resultado = puntos_y_resultado(registro["GolesFavor"], registro["GolesContra"])
        registro["Puntos"] = puntos
        registro["Resultado"] = resultado
        # Columnas 0/1 para contar victorias, empates y derrotas sumando.
        registro["Victoria"] = 1 if resultado == "V" else 0
        registro["Empate"] = 1 if resultado == "E" else 0
        registro["Derrota"] = 1 if resultado == "D" else 0
        registros.append(registro)
    return registros


def partidos_a_equipo_partido(tabla_partidos):
    """Aplica filas_de_un_partido a todos los partidos de una tabla.

    Returns:
        DataFrame con dos filas por partido.
    """
    registros = []
    for _, fila in tabla_partidos.iterrows():
        registros.extend(filas_de_un_partido(fila))
    return pd.DataFrame(registros)

### Prueba con datos de ejemplo controlados

Antes de usar las funciones con los datos reales, las probamos con tres partidos inventados cuyo resultado conocemos de antemano: una victoria del local, un empate y una victoria del visitante. Si alguna regla estuviera mal programada, un `assert` fallaría aquí.

In [ ]:
ejemplo = pd.DataFrame({
    "PartidoID": [0, 1, 2],
    "Temporada": ["2099/00"] * 3,
    "Date": pd.to_datetime(["2099-08-10", "2099-08-17", "2099-08-24"]),
    "HomeTeam": ["A", "B", "C"], "AwayTeam": ["B", "C", "A"],
    "FTHG": [2, 1, 0], "FTAG": [0, 1, 3],
    "HS": [10, 8, 5], "AS": [4, 9, 12], "HST": [5, 3, 1], "AST": [1, 3, 6],
    "HC": [6, 2, 3], "AC": [1, 5, 7], "HF": [9, 11, 10], "AF": [12, 10, 8],
    "HY": [1, 2, 0], "AY": [3, 1, 2], "HR": [0, 0, 1], "AR": [1, 0, 0],
})

prueba = partidos_a_equipo_partido(ejemplo)

# Partido 0: A gana 2-0 en casa a B.
assert prueba.loc[0, ["Equipo", "Puntos", "Resultado"]].tolist() == ["A", 3, "V"]
assert prueba.loc[1, ["Equipo", "Puntos", "Resultado"]].tolist() == ["B", 0, "D"]
# Partido 1: B y C empatan 1-1.
assert prueba.loc[2, "Puntos"] == 1 and prueba.loc[3, "Puntos"] == 1
# Partido 2: A gana 3-0 como visitante en casa de C.
assert prueba.loc[5, ["Equipo", "Condicion", "Puntos"]].tolist() == ["A", "Visitante", 3]
# Los goles y tiros del visitante salen de las columnas del visitante.
assert prueba.loc[5, ["GolesFavor", "GolesContra", "Tiros", "TirosArco"]].tolist() == [3, 0, 12, 6]

print("La prueba con datos controlados pasó.")
prueba[["PartidoID", "Equipo", "Rival", "Condicion", "GolesFavor", "GolesContra", "Puntos", "Resultado"]]

Ahora aplicamos las mismas funciones a los 1.900 partidos reales.

In [ ]:
equipo_partido = partidos_a_equipo_partido(partidos)
print(f"Registros equipo-partido: {len(equipo_partido)} (deben ser 2 × {len(partidos)})")
equipo_partido.head()

## Paso 2. Comprobar que la transformación es correcta

Verificamos con `assert` cuatro propiedades que deben cumplirse siempre:

1. Hay exactamente dos registros por partido.
2. **Los puntos repartidos en cada partido suman 3 si hubo ganador y 2 si hubo empate** (1 + 1). Ojo: en un empate no suman 3.
3. Los goles a favor del local son los goles en contra del visitante, y al revés.
4. Cada equipo juega 38 partidos por temporada: 19 de local y 19 de visitante.

In [ ]:
# 1. Dos registros por partido.
assert len(equipo_partido) == 2 * len(partidos)
assert (equipo_partido["PartidoID"].value_counts() == 2).all()

# 2. Puntos por partido: 3 si hay ganador, 2 si hay empate.
puntos_por_partido = equipo_partido.groupby("PartidoID")["Puntos"].sum()
puntos_esperados = partidos.set_index("PartidoID")["FTR"].map({"H": 3, "A": 3, "D": 2})
assert (puntos_por_partido == puntos_esperados).all()

# 3. Goles coherentes entre los dos equipos de cada partido.
local = equipo_partido[equipo_partido["Condicion"] == "Local"].set_index("PartidoID")
visitante = equipo_partido[equipo_partido["Condicion"] == "Visitante"].set_index("PartidoID")
assert (local["GolesFavor"] == visitante["GolesContra"]).all()
assert (local["GolesContra"] == visitante["GolesFavor"]).all()

# 4. 38 partidos por equipo y temporada: 19 de local y 19 de visitante.
conteo = equipo_partido.groupby(["Temporada", "Equipo", "Condicion"]).size()
assert (conteo == 19).all()

print("Las cuatro comprobaciones pasaron.")
print(f"Partidos con ganador: {(puntos_esperados == 3).sum()}, empates: {(puntos_esperados == 2).sum()}")

## Paso 3. Número de partido (1 a 38) y puntos acumulados

Como acordamos, no usamos la jornada oficial: los partidos aplazados hacen que el orden de las fechas no coincida con la jornada. En su lugar, numeramos los partidos de cada equipo **en orden de fecha**, del 1 al 38.

Con ese orden calculamos también los **puntos acumulados**: los puntos que el equipo llevaba después de cada partido. Los usaremos en el gráfico de evolución del dashboard.

Antes comprobamos que ningún equipo tenga dos partidos el mismo día; si los tuviera, el orden sería ambiguo.

In [ ]:
assert not equipo_partido.duplicated(["Temporada", "Equipo", "Fecha"]).any(), \
    "Un equipo tiene dos partidos el mismo día"

equipo_partido = equipo_partido.sort_values(["Temporada", "Equipo", "Fecha"]).reset_index(drop=True)

grupos = equipo_partido.groupby(["Temporada", "Equipo"])
equipo_partido["NumeroPartido"] = grupos.cumcount() + 1
equipo_partido["PuntosAcumulados"] = grupos["Puntos"].cumsum()

assert equipo_partido["NumeroPartido"].min() == 1
assert equipo_partido["NumeroPartido"].max() == 38

# Ejemplo: los primeros cinco partidos del Arsenal en 2023/24.
filtro = (equipo_partido["Temporada"] == "2023/24") & (equipo_partido["Equipo"] == "Arsenal")
equipo_partido.loc[filtro, ["NumeroPartido", "Fecha", "Rival", "Condicion",
                            "GolesFavor", "GolesContra", "Puntos", "PuntosAcumulados"]].head()

`groupby(["Temporada", "Equipo"])` separa los partidos de cada equipo en cada temporada. Dentro de cada grupo, `cumcount()` cuenta 0, 1, 2… (por eso sumamos 1) y `cumsum()` va sumando los puntos partido a partido.

## Paso 4. Estadísticas por equipo y temporada

Ahora sumamos los partidos de cada equipo en cada temporada. Lo hacemos **tres veces** con la misma función:

- **Total:** los 38 partidos.
- **Local:** solo los 19 partidos en casa.
- **Visitante:** solo los 19 partidos fuera.

Además de los totales, calculamos **promedios por partido**. Son los que permiten comparar de forma justa, por ejemplo, el rendimiento como local frente al de visitante.

**Sobre el dato vaciado en el notebook 1:** al West Ham 2021/22 le faltan los tiros de un partido como visitante. Por eso contamos aparte los partidos con dato de tiros (`PartidosConTiros`) y dividimos los tiros entre esa cantidad, no entre 38. Así su promedio no queda artificialmente bajo.

In [ ]:
COLUMNAS_A_SUMAR = [
    "Victoria", "Empate", "Derrota", "GolesFavor", "GolesContra", "Puntos",
    "Tiros", "TirosArco", "Corners", "Faltas", "Amarillas", "Rojas",
]


def resumir_equipos(registros):
    """Suma los registros equipo-partido de cada equipo en cada temporada.

    Args:
        registros: DataFrame en el nivel equipo-partido (puede estar filtrado).

    Returns:
        DataFrame con una fila por equipo y temporada.
    """
    grupos = registros.groupby(["Temporada", "Equipo"])
    resumen = grupos[COLUMNAS_A_SUMAR].sum()
    resumen.insert(0, "PJ", grupos.size())
    # count() no cuenta las celdas vacías: así sabemos en cuántos partidos hay dato de tiros.
    resumen["PartidosConTiros"] = grupos["Tiros"].count()
    resumen = resumen.reset_index()

    resumen = resumen.rename(columns={"Victoria": "V", "Empate": "E", "Derrota": "D"})
    resumen["DG"] = resumen["GolesFavor"] - resumen["GolesContra"]

    # Promedios por partido.
    resumen["PuntosPorPartido"] = resumen["Puntos"] / resumen["PJ"]
    resumen["GolesFavorPorPartido"] = resumen["GolesFavor"] / resumen["PJ"]
    resumen["GolesContraPorPartido"] = resumen["GolesContra"] / resumen["PJ"]
    resumen["TirosPorPartido"] = resumen["Tiros"] / resumen["PartidosConTiros"]
    resumen["TirosArcoPorPartido"] = resumen["TirosArco"] / resumen["PartidosConTiros"]
    resumen["CornersPorPartido"] = resumen["Corners"] / resumen["PJ"]
    resumen["FaltasPorPartido"] = resumen["Faltas"] / resumen["PJ"]
    resumen["AmarillasPorPartido"] = resumen["Amarillas"] / resumen["PJ"]
    resumen["RojasPorPartido"] = resumen["Rojas"] / resumen["PJ"]
    return resumen


# Diccionario: condición -> registros que le corresponden.
subconjuntos = {
    "Total": equipo_partido,
    "Local": equipo_partido[equipo_partido["Condicion"] == "Local"],
    "Visitante": equipo_partido[equipo_partido["Condicion"] == "Visitante"],
}

tablas = []
for condicion, registros in subconjuntos.items():
    tabla = resumir_equipos(registros)
    tabla.insert(2, "Condicion", condicion)
    tablas.append(tabla)

estadisticas = pd.concat(tablas, ignore_index=True)
print(f"Filas equipo-temporada: {len(estadisticas)} "
      f"(100 equipo-temporada × 3 condiciones)")
estadisticas.head()

### Comprobaciones de las estadísticas agregadas

1. Para cada equipo y temporada, **local + visitante = total** en todas las columnas que se suman.
2. Cada equipo tiene 38 partidos en total y 19 en cada condición.
3. En cada temporada, los goles a favor de todos los equipos suman lo mismo que los goles en contra, y las victorias lo mismo que las derrotas (cada gol y cada victoria de uno es el gol recibido y la derrota de otro).
4. Los tiros y tiros al arco solo tienen datos faltantes donde los vaciamos a propósito (West Ham 2021/22, un partido de visitante).

In [ ]:
indice = ["Temporada", "Equipo"]
total = estadisticas[estadisticas["Condicion"] == "Total"].set_index(indice)
de_local = estadisticas[estadisticas["Condicion"] == "Local"].set_index(indice)
de_visitante = estadisticas[estadisticas["Condicion"] == "Visitante"].set_index(indice)

# 1. Local + visitante = total.
for columna in ["PJ", "V", "E", "D", "GolesFavor", "GolesContra", "Puntos",
                "Tiros", "TirosArco", "Corners", "Faltas", "Amarillas", "Rojas"]:
    assert (de_local[columna] + de_visitante[columna] == total[columna]).all(), columna

# 2. Partidos jugados.
assert (total["PJ"] == 38).all()
assert (de_local["PJ"] == 19).all() and (de_visitante["PJ"] == 19).all()

# 3. Balance de cada temporada.
por_temporada = total.groupby("Temporada")[["GolesFavor", "GolesContra", "V", "D"]].sum()
assert (por_temporada["GolesFavor"] == por_temporada["GolesContra"]).all()
assert (por_temporada["V"] == por_temporada["D"]).all()

# 4. Partidos sin dato de tiros.
sin_dato = estadisticas[estadisticas["PartidosConTiros"] < estadisticas["PJ"]]
print("Equipo-temporada con algún partido sin dato de tiros:")
print(sin_dato[["Temporada", "Equipo", "Condicion", "PJ", "PartidosConTiros"]].to_string(index=False))
assert len(sin_dato) == 2  # West Ham 2021/22: Total y Visitante

print("\nTodas las comprobaciones pasaron.")

## Paso 5. Clasificación calculada

La clasificación se calcula **solo con los resultados**, sin copiarla de ninguna tabla externa. Ordenamos a los equipos de cada temporada con estos criterios, en este orden:

1. Más puntos.
2. Mejor diferencia de goles (`DG`).
3. Más goles a favor.

Si después de estos tres criterios dos equipos siguen empatados, **no inventamos un desempate**: la función los detecta y los muestra para documentarlo como limitación.

In [ ]:
CRITERIOS = ["Puntos", "DG", "GolesFavor"]


def calcular_clasificacion(estadisticas_total):
    """Ordena a los equipos de cada temporada por puntos, diferencia de goles y goles a favor.

    Args:
        estadisticas_total: filas con Condicion == "Total".

    Returns:
        DataFrame con la columna "Posicion" (1 a 20 en cada temporada).
    """
    tabla = estadisticas_total.sort_values(
        ["Temporada"] + CRITERIOS, ascending=[True, False, False, False]
    ).reset_index(drop=True)
    tabla.insert(0, "Posicion", tabla.groupby("Temporada").cumcount() + 1)
    return tabla


def empates_sin_resolver(clasificacion):
    """Devuelve los equipos que siguen empatados después de los tres criterios."""
    repetidos = clasificacion.duplicated(["Temporada"] + CRITERIOS, keep=False)
    return clasificacion[repetidos]


clasificacion = calcular_clasificacion(estadisticas[estadisticas["Condicion"] == "Total"])

empates = empates_sin_resolver(clasificacion)
if len(empates) == 0:
    print("No hay empates sin resolver: los tres criterios ordenan a todos los equipos.")
else:
    print("Empates que necesitan un criterio adicional (se documentan como limitación):")
    print(empates[["Temporada", "Posicion", "Equipo"] + CRITERIOS].to_string(index=False))

columnas_tabla = ["Posicion", "Equipo", "PJ", "V", "E", "D", "GolesFavor", "GolesContra", "DG", "Puntos"]
clasificacion[clasificacion["Temporada"] == "2024/25"][columnas_tabla]

## Paso 6. Verificación contra la tabla oficial de 2023/24

Para comprobar que los cálculos son correctos, comparamos nuestra tabla de 2023/24 con la tabla final publicada por NBC Sports (https://www.nbcsports.com/soccer/premier-league-2023-24-table). Elegimos esa temporada porque es la única de las cinco con **sanciones de puntos**:

- **Everton:** −8 puntos (−6 por un primer incumplimiento de las reglas financieras, reducido en apelación desde −10, y −2 por un segundo).
- **Nottingham Forest:** −4 puntos.

Las sanciones no salen de los partidos, así que nuestra tabla no las incluye. Para comparar, a nuestros puntos les restamos la sanción y el resultado debe coincidir con los puntos oficiales.

In [ ]:
# Tabla oficial 2023/24 (NBC Sports): equipo -> (V, E, D, GF, GC, puntos oficiales, sanción).
OFICIAL_2324 = {
    "Man City": (28, 7, 3, 96, 34, 91, 0),
    "Arsenal": (28, 5, 5, 91, 29, 89, 0),
    "Liverpool": (24, 10, 4, 86, 41, 82, 0),
    "Aston Villa": (20, 8, 10, 76, 61, 68, 0),
    "Tottenham": (20, 6, 12, 74, 61, 66, 0),
    "Chelsea": (18, 9, 11, 77, 63, 63, 0),
    "Newcastle": (18, 6, 14, 85, 62, 60, 0),
    "Man United": (18, 6, 14, 57, 58, 60, 0),
    "West Ham": (14, 10, 14, 60, 74, 52, 0),
    "Crystal Palace": (13, 10, 15, 57, 58, 49, 0),
    "Brighton": (12, 12, 14, 55, 62, 48, 0),
    "Bournemouth": (13, 9, 16, 54, 67, 48, 0),
    "Fulham": (13, 8, 17, 55, 61, 47, 0),
    "Wolves": (13, 7, 18, 50, 65, 46, 0),
    "Everton": (13, 9, 16, 40, 51, 40, 8),
    "Brentford": (10, 9, 19, 56, 65, 39, 0),
    "Nott'm Forest": (9, 9, 20, 49, 67, 32, 4),
    "Luton": (6, 8, 24, 52, 85, 26, 0),
    "Burnley": (5, 9, 24, 41, 78, 24, 0),
    "Sheffield United": (3, 7, 28, 35, 104, 16, 0),
}

calculada = clasificacion[clasificacion["Temporada"] == "2023/24"].set_index("Equipo")
diferencias = []
for equipo, (v, e, d, gf, gc, puntos_oficiales, sancion) in OFICIAL_2324.items():
    fila = calculada.loc[equipo]
    nuestros = (fila["V"], fila["E"], fila["D"], fila["GolesFavor"], fila["GolesContra"])
    if nuestros != (v, e, d, gf, gc) or fila["Puntos"] - sancion != puntos_oficiales:
        diferencias.append(equipo)

assert len(calculada) == len(OFICIAL_2324) == 20
assert diferencias == [], f"No coinciden: {diferencias}"
print("Los 20 equipos coinciden con la tabla oficial de 2023/24 (descontando las sanciones).")

for equipo in ["Everton", "Nott'm Forest"]:
    puntos = calculada.loc[equipo, "Puntos"]
    print(f"{equipo}: {puntos} puntos en el campo, "
          f"{OFICIAL_2324[equipo][5]} oficiales tras la sanción de {OFICIAL_2324[equipo][6]}.")

**Qué significa esto para el proyecto:** nuestra pregunta guía estudia cómo se relaciona el juego (tiros, córners, faltas, tarjetas) con los puntos. Para eso, los puntos que importan son los **ganados en el campo**, que son los que calculamos. Las sanciones administrativas no dependen del juego.

En el dashboard habrá que avisar que, en 2023/24, la clasificación de Everton y Nottingham Forest difiere de la oficial por este motivo.

## Paso 7. Guardar las tablas

Guardamos tres archivos, uno por nivel de análisis. Los usarán el notebook de análisis exploratorio y la aplicación de Streamlit.

In [ ]:
equipo_partido.to_csv("equipo_partido.csv", index=False)
estadisticas.to_csv("estadisticas_equipo_temporada.csv", index=False)
clasificacion.to_csv("clasificacion.csv", index=False)

# Comprobamos que se pueden volver a leer con el mismo tamaño.
for nombre, tabla in [("equipo_partido.csv", equipo_partido),
                      ("estadisticas_equipo_temporada.csv", estadisticas),
                      ("clasificacion.csv", clasificacion)]:
    leida = pd.read_csv(nombre)
    assert leida.shape == tabla.shape, nombre
    print(f"{nombre}: {leida.shape[0]} filas y {leida.shape[1]} columnas")

## Para terminar

1. Descarguen los tres CSV desde el panel de archivos (clic derecho → Descargar). Irán a `data/processed/` en el repositorio.
2. El siguiente notebook usará `estadisticas_equipo_temporada.csv` para las estadísticas descriptivas y las correlaciones.

**Preguntas que deben poder responder en la sustentación**

- ¿Por qué cada partido se convierte en dos filas?
- ¿Por qué los puntos de un partido suman 3 con ganador pero 2 con empate?
- ¿Qué comprueba la igualdad "local + visitante = total"?
- ¿Por qué los goles a favor de una temporada suman lo mismo que los goles en contra?
- ¿Por qué los tiros por partido del West Ham 2021/22 se dividen entre 37 y no entre 38 en el total?
- ¿Por qué nuestra clasificación de 2023/24 no coincide exactamente con la oficial, y por qué usamos los puntos ganados en el campo?
- ¿Qué hace el programa si dos equipos empatan en puntos, diferencia de goles y goles a favor?